# Overnight experiment: the whole run as a notebook

Same blocks as `run_overnight.py`, one cell per stage, so you can run, look, and re-run.

**Order:** 1 settings → 2 run object → 3 checks → 4 map store + geometry → 5 F4 learned features → 6 candidate dump (the long one) → 7 fusion + ranker → 8 full-map evaluation on NUDT test → 9 **official development set** (overfitting check) → 10 report.

* Every stage is saved in `results/overnight/state.json`; running a cell again skips what is finished (pass `redo=["f4"]` to force).
* `SMOKE = True` runs everything on a tiny synthetic world (about 15 min on CPU) just to prove the plumbing.
* The F4 weights from the interrupted run are reused automatically if `results/overnight/models/f4.pt` exists.
* If the kernel dies from RAM: restart it, run cells 1-2, then only the stage you want. Finished work is kept on disk.

In [1]:
# 1. settings
import os, sys, json, argparse
from pathlib import Path

SMOKE        = False          # True = tiny synthetic run, False = real NUDT
HOURS        = 10.0           # total time budget (the candidate dump uses about 55 % of it)
N_TRAIN      = 3000           # photos to learn from
N_VALID      = 800            # photos to choose with
N_TEST       = 400            # photos for the NUDT test report
N_OFFICIAL   = 500            # official dev-set photos to test (500 = all)
F4_MINUTES   = 90             # learned-features training time (ignored if f4.pt exists)
# os.environ["NUDT_DIR"] = "/path/to/dataset/NUDT"        # only if NUDT is not at ../dataset/NUDT
# os.environ["DEV_DATA_DIR"] = "/path/to/dataset"         # only if d1/ and small_sat_area/ are not at ../dataset

ROOT = Path.cwd()
while not ((ROOT / "train").exists() and (ROOT / "src").exists()):
    ROOT = ROOT.parent                    # walk up to the repo folder
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("working in", ROOT)

working in /home/voidness/Storage/Synced Folders/InterIIT Tech 15/Code_Testing/test_src


In [2]:
# 2. create the run object
import sys
sys.path.insert(0, str(ROOT / "tests" / "Test_Iteration_1"))
import pandas as pd
from IPython.display import display, Markdown
from train import config
import run_overnight
from stages.driver import run_stages
from stages.run_context import RunContext

args = argparse.Namespace(smoke=SMOKE, hours=HOURS, stages=None, redo=[], out=str(config.OUT_DIR),
                          n_train=N_TRAIN, n_valid=N_VALID, n_test=N_TEST, n_official=N_OFFICIAL, f4_minutes=F4_MINUTES)
if SMOKE:
    args = run_overnight.prepare_smoke(args)
run = RunContext(args, args.out, hours=args.hours)

def go(*stages, redo=()):
    """Run some stages (setup is added by itself when needed and is built only once per kernel)."""
    run.args.stages = list(stages)
    run.args.redo = list(redo)
    run_stages(run, run_overnight.STAGE_FUNCTIONS, list(stages), list(redo), run_overnight.NEEDS_SETUP)

def show(key):
    rows = run.state["results"].get(key, [])
    if rows:
        display(pd.DataFrame([{"map / model": r["map"], "top-1 <= 1 sigma": round(r["top1_within_1sigma"], 3),
                               "<= 2 sigma": round(r["top1_within_2sigma"], 3), "<= 4 sigma": round(r["top1_within_4sigma"], 3),
                               "best of 25 <= 4 sigma": round(r["ceiling25_within_4sigma"], 3),
                               "change vs edge_zncc": round(r["vs_baseline_4sigma"]["diff"], 3),
                               "95% low": round(r["vs_baseline_4sigma"]["lo"], 3), "95% high": round(r["vs_baseline_4sigma"]["hi"], 3)} for r in rows]))

print("output folder:", run.out, "| stages already done:", run.state["done"])

In [3]:
# 3. checks: GPU, packages, NUDT files
go("check")

[09:29:57 +  0.0 min] == stage check
[09:29:57 +  0.0 min] torch 2.14.0+cu130 | device: cuda | gpu: NVIDIA GeForce RTX 3050 6GB Laptop GPU
[09:29:57 +  0.0 min] NUDT folder: /home/voidness/Storage/Synced Folders/InterIIT Tech 15/Code_Testing/dataset/NUDT | map file: /home/voidness/Storage/Synced Folders/InterIIT Tech 15/Code_Testing/dataset/NUDT/Reference_map/changsha/ref.tif | 3.4 GB
[09:29:57 +  0.0 min]   train:  8115 frames in 14 flights
[09:29:58 +  0.0 min]   valid:  1425 frames in 5 flights
[09:29:58 +  0.0 min]   test :  2350 frames in 4 flights
[09:29:58 +  0.0 min] == stage check took 0.0 min
[09:29:58 +  0.0 min] finished. failed stages: none


In [4]:
# 4. photos, camera, map store (a few minutes), then the photo-to-map geometry
go("geometry")
print(run.state["results"].get("geometry"))

[09:29:58 +  0.0 min] == stage setup
[09:30:11 +  0.2 min] map: 40961 x 27770 px, 0.30 m/px
[09:30:11 +  0.2 min]   train: 3000 photos from 14 flights
[09:30:11 +  0.2 min]   valid: 800 photos from 5 flights
[09:30:11 +  0.2 min]   test : 400 photos from 4 flights
[09:30:11 +  0.2 min] building the map store (levels [3, 4]) ...
level 3: 0.85 m/pixel, picture 14481 x 9818, map coverage 96%
level 4: 1.20 m/pixel, picture 10240 x 6942, map coverage 96%
map store ready, took 48 s
[09:30:59 +  1.0 min] == stage setup took 1.0 min
[09:30:59 +  1.0 min] -- stage geometry: already done, skipping
[09:30:59 +  1.0 min] finished. failed stages: none
{'attitude': False, 'yaw_offset': 0.0, 'scale': 0.8, 'height_offset': 0.0, 'yaw_sign': 1, 'share_within_4_sigma': 0.75}


### F4: learned features
Trains two small networks so that the correlation of their cell grids peaks at the true position. The log compares them with plain edge features on held-out pairs. If `models/f4.pt` already exists this cell only loads it.

In [5]:
go("f4")   # to retrain from scratch: delete results/overnight/models/f4.pt, then go("f4", redo=["f4"])
print(run.state["results"].get("f4"))

[09:30:59 +  1.0 min] -- stage f4: already done, skipping
[09:30:59 +  1.0 min] finished. failed stages: none
None


### Candidate dump (the long stage)
For every train and valid photo: all score maps → one small training example. Restart-safe: finished shards are skipped. If you change the photo counts or the list of maps, delete `results/overnight/shards_*` first.

In [6]:
go("dump")

[09:30:59 +  1.0 min] -- stage dump: already done, skipping
[09:30:59 +  1.0 min] finished. failed stages: none


### Fusion and ranker
Trains linear / monotone / gated fusion on several map subsets and compares each with the plain edge ZNCC (paired bootstrap).

In [7]:
go("fusion", "ranker")
rows = run.state["results"].get("fusion_rows", [])
if rows:
    display(pd.DataFrame([{"maps": r["maps"], "model": r["kind"], "valid top-1": round(r["valid_top1"], 3),
                           "vs edge_zncc": round(r["vs_edge_zncc"]["diff"], 3), "95% low": round(r["vs_edge_zncc"]["lo"], 3),
                           "95% high": round(r["vs_edge_zncc"]["hi"], 3)} for r in rows]))
print("best:", run.state["results"].get("fusion_best", {}).get("tag"))

[09:30:59 +  1.0 min] -- stage fusion: already done, skipping
[09:30:59 +  1.0 min] -- stage ranker: already done, skipping
[09:30:59 +  1.0 min] finished. failed stages: none


,maps,model,valid top-1,vs edge_zncc,95% low,95% high
0,edge_zncc,monotone,0.124,0.000,0.000,0.000
1,edge_zncc,gated,0.124,0.000,0.000,0.000
2,edge_zncc+edge_cosine+edge_mag,linear,0.204,0.080,0.044,0.120
3,edge_zncc+edge_cosine+edge_mag,monotone,0.200,0.076,0.040,0.116
4,edge_zncc+edge_cosine+edge_mag,gated,0.200,0.076,0.040,0.116
5,edge_zncc+edge_cosine+edge_mag+edge_zncc_w,linear,0.208,0.084,0.048,0.124
6,edge_zncc+edge_cosine+edge_mag+edge_zncc_w,monotone,0.176,0.052,0.016,0.092
7,edge_zncc+edge_cosine+edge_mag+edge_zncc_w,gated,0.204,0.080,0.040,0.124
8,edge_zncc+edge_cosine+edge_mag+mind_zncc,linear,0.200,0.076,0.040,0.116
9,edge_zncc+edge_cosine+edge_mag+mind_zncc,monotone,0.176,0.052,0.016,0.092


best: edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc__monotone


### Full-map evaluation on NUDT test flights
The real search over the whole 5 km disc, for photos never used to learn or choose.

In [8]:
go("evaluate")
show("evaluate")

[09:30:59 +  1.0 min] -- stage evaluate: already done, skipping
[09:30:59 +  1.0 min] finished. failed stages: none


,map / model,top-1 <= 1 sigma,<= 2 sigma,<= 4 sigma,best of 25 <= 4 sigma,change vs edge_zncc,95% low,95% high
0,edge_zncc,0.053,0.100,0.173,0.493,0.000,0.000,0.000
1,edge_cosine,0.033,0.080,0.153,0.480,-0.020,-0.073,0.033
2,edge_mag,0.000,0.000,0.007,0.100,-0.167,-0.227,-0.107
3,edge_zncc_w,0.020,0.033,0.087,0.293,-0.087,-0.133,-0.040
4,mind_zncc,0.040,0.060,0.120,0.373,-0.053,-0.113,0.007
5,learned_zncc,0.073,0.193,0.260,0.553,0.087,0.020,0.153
6,fused:edge_zncc+edge_cosine+edge_mag+edge_zncc...,0.073,0.160,0.300,0.633,0.127,0.067,0.193
7,fused:edge_zncc+edge_cosine+edge_mag+edge_zncc...,0.087,0.193,0.307,0.627,0.133,0.080,0.200


### Official development set (the overfitting / domain check)
Same trained models, on the 500 official photos and the `small_sat_area` map: another city, camera, sensor and map. This frees the NUDT map from memory first, so run it after the NUDT stages (or in a fresh kernel: cells 1-2, then this cell).
If a fused model beats `edge_zncc` on NUDT test but not here, the gain belonged to Changsha.

In [9]:
go("official")
show("official")

[09:30:59 +  1.0 min] == stage official
[09:31:13 +  1.3 min] official set: 379 photos tested of 500, map 22096 x 13840 px at 0.30 m/px, geometry {'attitude': True, 'pitch_sign': -1, 'roll_sign': -1, 'yaw_offset': 180.0, 'scale': 1.04, 'height_offset': 0.0}
level 3: 0.85 m/pixel, picture 7812 x 4893, map coverage 70%
level 4: 1.20 m/pixel, picture 5524 x 3460, map coverage 70%
level 5: 1.70 m/pixel, picture 3906 x 2446, map coverage 70%
level 6: 2.40 m/pixel, picture 2762 x 1730, map coverage 70%
level 7: 3.39 m/pixel, picture 1953 x 1223, map coverage 70%
map store ready, took 21 s
[09:31:51 +  1.9 min] learned family loaded (valid hit rate 93.3 % vs edge 90.5 %)
[09:31:53 +  1.9 min] full-map check: 1 / 379 photos, 2 s
[09:32:07 +  2.2 min] full-map check: 21 / 379 photos, 15 s
[09:32:19 +  2.4 min] full-map check: 41 / 379 photos, 28 s
[09:32:34 +  2.6 min] full-map check: 61 / 379 photos, 42 s
[09:32:47 +  2.8 min] full-map check: 81 / 379 photos, 56 s
[09:33:04 +  3.1 min] full-ma

,map / model,top-1 <= 1 sigma,<= 2 sigma,<= 4 sigma,best of 25 <= 4 sigma,change vs edge_zncc,95% low,95% high
0,edge_zncc,0.554,0.789,0.805,0.900,0.000,0.000,0.000
1,edge_cosine,0.525,0.763,0.778,0.889,-0.026,-0.055,0.003
2,edge_mag,0.201,0.301,0.306,0.562,-0.499,-0.551,-0.446
3,edge_zncc_w,0.536,0.755,0.768,0.855,-0.037,-0.061,-0.013
4,mind_zncc,0.557,0.792,0.815,0.942,0.011,-0.024,0.045
5,learned_zncc,0.528,0.839,0.863,0.953,0.058,0.029,0.087
6,fused:edge_zncc+edge_cosine+edge_mag+edge_zncc...,0.628,0.889,0.900,0.963,0.095,0.066,0.127
7,fused:edge_zncc+edge_cosine+edge_mag+edge_zncc...,0.615,0.879,0.894,0.955,0.090,0.061,0.121


In [10]:
# 10. the report
go("report")
Markdown((run.out / "report.md").read_text())

[09:36:32 +  6.6 min] == stage report
[09:36:32 +  6.6 min] report written: /home/voidness/Storage/Synced Folders/InterIIT Tech 15/Code_Testing/test_src/overnight_out/report.md
[09:36:32 +  6.6 min] == stage report took 0.0 min
[09:36:32 +  6.6 min] finished. failed stages: none


# Overnight run report

Written 2026-11-04 09:36. Mode: NUDT.

## Photo-to-map geometry

yaw sign +1, yaw offset 0 deg, scale 0.80; 75.0 % of the check photos land within 4 sigma with it.
If this is below 30 %% nothing downstream can be trusted.

## C2, fusion (valid flights, truth against the candidate list)

| maps | model | valid top-1 | vs edge_zncc alone (95 % interval) |
|---|---|---|---|
| edge_zncc | monotone | 0.124 | +0.000 [+0.000, +0.000] |
| edge_zncc | gated | 0.124 | +0.000 [+0.000, +0.000] |
| edge_zncc+edge_cosine+edge_mag | linear | 0.204 | +0.080 [+0.044, +0.120] |
| edge_zncc+edge_cosine+edge_mag | monotone | 0.200 | +0.076 [+0.040, +0.116] |
| edge_zncc+edge_cosine+edge_mag | gated | 0.200 | +0.076 [+0.040, +0.116] |
| edge_zncc+edge_cosine+edge_mag+edge_zncc_w | linear | 0.208 | +0.084 [+0.048, +0.124] |
| edge_zncc+edge_cosine+edge_mag+edge_zncc_w | monotone | 0.176 | +0.052 [+0.016, +0.092] |
| edge_zncc+edge_cosine+edge_mag+edge_zncc_w | gated | 0.204 | +0.080 [+0.040, +0.124] |
| edge_zncc+edge_cosine+edge_mag+mind_zncc | linear | 0.200 | +0.076 [+0.040, +0.116] |
| edge_zncc+edge_cosine+edge_mag+mind_zncc | monotone | 0.176 | +0.052 [+0.016, +0.092] |
| edge_zncc+edge_cosine+edge_mag+mind_zncc | gated | 0.192 | +0.068 [+0.028, +0.112] |
| edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc | linear | 0.200 | +0.076 [+0.040, +0.116] |
| edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc | monotone | 0.172 | +0.048 [+0.012, +0.084] |
| edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc | gated | 0.188 | +0.064 [+0.028, +0.104] |
| edge_zncc+learned_zncc | linear | 0.192 | +0.068 [+0.036, +0.104] |
| edge_zncc+learned_zncc | monotone | 0.192 | +0.068 [+0.032, +0.104] |
| edge_zncc+learned_zncc | gated | 0.200 | +0.076 [+0.032, +0.120] |
| edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc | linear | 0.248 | +0.124 [+0.084, +0.172] |
| edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc | monotone | 0.252 | +0.128 [+0.084, +0.176] |
| edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc | gated | 0.236 | +0.112 [+0.068, +0.156] |

Best on valid: **edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc / monotone** (0.252).

## R1, ranker on the hard candidates

fused score alone 0.252, ranker 0.536, difference +0.284 (95 % +0.208 .. +0.356). Use it only if the interval is above 0.

## Final check on the full score maps (NUDT test flights, never used for choosing)

| map / model | top-1 within 1 sigma | 2 sigma | 4 sigma | best of 25 within 4 sigma | median error (sigma) | 4-sigma change vs edge_zncc |
|---|---|---|---|---|---|---|
| edge_zncc | 0.053 | 0.100 | 0.173 | 0.493 | 165.1 | +0.000 [+0.000, +0.000] |
| edge_cosine | 0.033 | 0.080 | 0.153 | 0.480 | 198.2 | -0.020 [-0.073, +0.033] |
| edge_mag | 0.000 | 0.000 | 0.007 | 0.100 | 358.4 | -0.167 [-0.227, -0.107] |
| edge_zncc_w | 0.020 | 0.033 | 0.087 | 0.293 | 230.5 | -0.087 [-0.133, -0.040] |
| mind_zncc | 0.040 | 0.060 | 0.120 | 0.373 | 210.3 | -0.053 [-0.113, +0.007] |
| learned_zncc | 0.073 | 0.193 | 0.260 | 0.553 | 159.8 | +0.087 [+0.020, +0.153] |
| fused:edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc__monotone | 0.073 | 0.160 | 0.300 | 0.633 | 113.6 | +0.127 [+0.067, +0.193] |
| fused:edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc__linear | 0.087 | 0.193 | 0.307 | 0.627 | 96.0 | +0.133 [+0.080, +0.200] |

Pass rule C2 (fused >= best single by 0.03 at the 25-candidate ceiling): best fused fused:edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc__monotone vs best single learned_zncc (55.3 %): gain +0.080 -> **PASS**.

## Official development set (d1 photos, small_sat_area map): the overfitting / domain check

Models were fitted on NUDT only. Same score maps and fusion models, different city, camera, sensor and map.

| map / model | NUDT test 4-sigma | official 4-sigma | official 1-sigma | official best of 25 | official change vs edge_zncc (95 % interval) |
|---|---|---|---|---|---|
| edge_zncc | 0.173 | 0.805 | 0.554 | 0.900 | +0.000 [+0.000, +0.000] |
| edge_cosine | 0.153 | 0.778 | 0.525 | 0.889 | -0.026 [-0.055, +0.003] |
| edge_mag | 0.007 | 0.306 | 0.201 | 0.562 | -0.499 [-0.551, -0.446] |
| edge_zncc_w | 0.087 | 0.768 | 0.536 | 0.855 | -0.037 [-0.061, -0.013] |
| mind_zncc | 0.120 | 0.815 | 0.557 | 0.942 | +0.011 [-0.024, +0.045] |
| learned_zncc | 0.260 | 0.863 | 0.528 | 0.953 | +0.058 [+0.029, +0.087] |
| fused:edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc__monotone | 0.300 | 0.900 | 0.628 | 0.963 | +0.095 [+0.066, +0.127] |
| fused:edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc__linear | 0.307 | 0.894 | 0.615 | 0.955 | +0.090 [+0.061, +0.121] |

Reading: if a fused model beats edge_zncc on NUDT test but its interval on the official set includes 0 or is negative, the gain was specific to Changsha (overfitting to the domain). Best fused here: fused:edge_zncc+edge_cosine+edge_mag+edge_zncc_w+mind_zncc+learned_zncc__monotone, change +0.095 (lower end +0.066) -> **holds on the official set**.

## Files

run.log (everything that happened), state.json (all numbers), models/ (f4.pt, fusion/*.pt, ranker.pt), shards_*/ (candidate sets).

In [11]:
# anything that failed is listed here with the reason (details in results/overnight/run.log)
print(json.dumps(run.state["failed"], indent=1) if run.state["failed"] else "nothing failed")

nothing failed
